
### Завдання 1: Виклик LLM з базовим промптом

Створіть можливість викликати LLM зі звичайним текстовим промптом.

Промпт має дозвляти отримати інформацію простою мовою на певну тему. В цьому завданні ми хочемо дізнатись про тему "Квантові обчислення".

Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

Обмежте відповідь до 200 символів і пропишіть в промпті аби відповідь була короткою (це зекономить Вам час і гроші на згенеровані токени).

В якості LLM можна скористатись як моделлю з HugginFace (рекомендую Mistral), так і ChatGPT4 або ChatGPT3. В обох випадках треба імпортувати потрібну "обгортку" (тобто клас, який дозволить ініціювати модель) з LangChain для виклику LLM за API, а також зчитати особистий токен з файла, наприклад, `creds.json`, який розміщений у Вас локально і Ви НЕ здаєте його в ДЗ і НЕ комітите в git 😏

Встановіть своє значення температури на свій розсуд (тут немає правильного чи неправильного значення) і напишіть, чому ви обрали саме таке значення для цього завдання.  

Запити можна робити як українською, так і англійською - орієнтуйтесь на те, де і чи хочете ви потім лишити цей проєкт і відповідна яка мова буде пасувати більше. В розвʼязках промпти - українською.

In [1]:
!pip install langchain-mistralai -q

In [2]:
import json
import os
from langchain.prompts import PromptTemplate
from langchain_mistralai import ChatMistralAI

In [3]:
with open('creds.json') as file:
  creds = json.load(file)

os.environ['MISTRAL_API_KEY'] = creds['mistral_api_key']

In [4]:
overall_temperature = 0.1

In [5]:
chat_mistral = ChatMistralAI(
    model="mistral-large-latest",
    temperature=overall_temperature,
    max_tokens=200
)

In [6]:
prompt = (
    "Відповідай коротко (до 200 символів)."
    "Тема: квантові обчислення."
    "Дай визначення, ключові переваги та приклади поточних досліджень."
    "Відповідай лише українською мовою"
)

In [9]:
response = chat_mistral.invoke(prompt)
print(response.content)

**Квантові обчислення** – використання квантових бітів (кубітів), що можуть бути в суперпозиції станів (0 та 1 одночасно), для прискорення обчислень.

**Переваги**:
- Експоненціальне прискорення для певних задач (факторизація, симуляція молекул).
- Паралелізм завдяки квантовій заплутаності.

**Дослідження 2023–2024**:
- **Google/IBM**: Покращення квантової корекції помилок (1000+ логічних кубітів).
- **China**: Квантовий супутник *Micius* для захищених комунікацій.
- **Startups (Rigetti, IonQ)**: Гібридні к


### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [10]:
template = """
Відповідай коротко (до 200 символів).
Тема: {topic}.
Дай визначення, ключові переваги та приклади поточних досліджень згідно теми.
Відповідай українською мовою.
"""

In [11]:
prompt_template = PromptTemplate(
    input_variables=["topic"],
    template=template,
)

In [12]:
topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI"
]

In [13]:
for t in topics:
    prompt = prompt_template.format(topic=t)
    resp = chat_mistral.invoke(prompt)
    print(f"Тема: {t}\nВідповідь: {resp.content}\n{'-'*80}")

Тема: Баєсівські методи в машинному навчанні
Відповідь: **Баєсівські методи** — підхід у ML, що використовує теорему Баєса для оновлення ймовірностей на основі даних, враховуючи апріорні знання.

**Переваги**:
- Врахування невизначеності.
- Ефективність при малих даних.
- Інтерпретованість результатів.

**Приклади досліджень**:
- **Баєсівські нейронні мережі** (UCL, 2023) для надійнішого DL.
- **MCMC-оптимізація** в генеративних моделях (Stanford, 2024).
- **Федеративне навчання** з баєсівською агрегацією (Google Research, 2023).
--------------------------------------------------------------------------------
Тема: Трансформери в машинному навчанні
Відповідь: **Трансформери** – архітектура глибокого навчання (2017, Vaswani et al.), що базується на **механізмі уваги** (*self-attention*) для обробки послідовностей (текст, зображення, звук).

**Переваги**:
✔ Паралелізація обчислень (швидке навчання).
✔ Далекі залежності в даних (краще за RNN/LSTM).
✔ Універсальність (NLP, комп’ютерний зір



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [16]:
!pip install -q google-search-results langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [17]:
from langchain.agents import load_tools
from langchain.agents import Tool, AgentExecutor, AgentType, create_react_agent, initialize_agent
from langchain.agents import create_tool_calling_agent
from langchain import hub

In [18]:
chat_mistral_without_limit = ChatMistralAI(
    model="mistral-large-latest",
    temperature=overall_temperature
)

In [19]:
os.environ['SERPAPI_API_KEY'] = creds['SERPAPI_API_KEY']

In [20]:
tools = load_tools(["serpapi"], llm=chat_mistral)

In [21]:
hub_prompt = hub.pull("hwchase17/openai-tools-agent")

In [22]:
agent = create_tool_calling_agent(chat_mistral_without_limit, tools, hub_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [29]:
agent_executor.invoke({
    'input':
    '''
      Знайди в інтернеті або в базах даних наукових публікацій 5 останніх публікацій на тему прогнозування врожайності.
      Видай список публікацій, кожна з яких містить назву публікацій, авторів і короткий опис.'
    '''
})



> Entering new AgentExecutor chain...

Invoking: `Search` with `останні наукові публікації на тему прогнозування врожайності 2023-2024`


['Стаття підготовлена в рамках виконання науково-дослідної роботи на тему «Прогнозування розвитку ринку зерно- вих нішевих культур в умовах ...', 'Прогноз світового виробництва фуражного зерна у 2024 році цього місяця був дещо знижений (до 1521 млн тонн), що на 0,8 відсотки менше за показник попереднього ...', 'Тоді ж у НААН висували прогнози, що сезон-2023 завершиться з результатом 53,3 млн тонн, і це з урахуванням прогнозованого скорочення посівних ...', 'Валове виробництво олійних культур може сягнути 21,6 млн т. Врожай соняшнику прогнозується на рівні 13 млн т. Ріпаку, як прогнозується, аграрії ...', 'сезоні 2023–2024 рр. за прогнозами USDA черговий рекорд – 789,8 млн т [16].Рейтинг країн-виробників у 2022–2023 роках очолю- вав Китай – 138 млн т (рис.2) ...', 'Члени оргкомітету: Черенков А. В. – доктор с.-г. наук, професор, академік НААН;. Дзюб

{'input': "\n      Знайди в інтернеті або в базах даних наукових публікацій 5 останніх публікацій на тему прогнозування врожайності.\n      Видай список публікацій, кожна з яких містить назву публікацій, авторів і короткий опис.'\n    ",
 'output': 'Ось список 5 останніх наукових публікацій на тему прогнозування врожайності з назвою, авторами та коротким описом:\n\n---\n\n1. **Назва:** *Crop yield prediction: An operational approach to crop yield modeling on field and subfield level with machine learning models*\n   **Автори:** P. Helber, B. Bischke, P. Habelitz\n   **Опис:**\n   У роботі запропоновано операційний підхід до моделювання врожайності сільськогосподарських культур на рівні поля та субполя з використанням машинного навчання. Дослідження демонструє, як моделі машинного навчання можуть бути інтегровані в практичні системи для точнішого прогнозування врожайності.\n\n---\n\n2. **Назва:** *A comparative analysis of the machine learning model for crop yield prediction in Quezon P



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [31]:
!pip install -q langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 5.6 MB/s eta 0:00:00


In [41]:
from langchain_experimental.utilities import PythonREPL
from langchain_core.prompts import ChatPromptTemplate

In [42]:
python_repl = PythonREPL()
python_tool = Tool(
    name="python_repl",
    description="A Python shell. Use this to execute python commands. Input should be a valid python command. If you want to see the output of a value, you necessarily should print it out with `print(...)`. Otherwise you won't see the result! It's very important.",
    func=python_repl.run,
)
python_tool.name = "python_interpreter"

In [43]:
tools.append(python_tool)

In [48]:
my_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Ти — професійний бізнес-аналітик. Твоє завдання — створити прогноз продажів на основі історичних даних та зовнішніх факторів, знайдених в інтернеті.

            Чітко дотримуйся цих кроків:
            1.  **Проаналізуй історичні дані**: Визначте обсяги експорту та роки з запиту користувача.
            2.  **Розрахуй базовий тренд**: Використовуй `python_interpreter` для розрахунку простого тренду або середнього зростання на основі історичних даних. Це буде твій початковий прогноз.
            3.  **Досліди зовнішні фактори**:
                а.  Знайди прогноз погоди для регіонів вирощування апельсинів у Бразилії на наступний рік (2025).
                б.  Досліди глобальний економічний прогноз, звертаючи увагу на споживчий попит та інфляцію.
            4.  **Узагальни дані та скоригуй прогноз**:
                а.  На основі прогнозу погоди визнач відсоткове коригування (наприклад, -10% за очікувану посуху, +5% за сприятливі умови).
                б.  На основі економічного прогнозу визнач друге відсоткове коригування (наприклад, -5% за високу інфляцію, +3% за сильну економіку).
            5.  **Зроби фінальний розрахунок**: Використовуй `python_interpreter`, щоб застосувати обидва відсоткові коригування до твого базового прогнозу з кроку 2.
            6.  **Надай остаточну відповідь**: Представ фінальну прогнозовану цифру. Чітко поясни свою логіку та методологію, посилаючись на знайдені погодні та економічні фактори. Якщо ти не можеш знайти достовірну інформацію для коригування, повідом про це і надай базовий прогноз, пояснивши, яких даних не вистачає.
            """,
        ),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

In [49]:
agent = create_tool_calling_agent(chat_mistral_without_limit, tools, my_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [50]:
agent_executor.invoke({
    'input':
    '''
      Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
    '''
})



> Entering new AgentExecutor chain...

Invoking: `python_interpreter` with `# Історичні дані про експорт апельсинів (в тонах)
years = [2021, 2022, 2023, 2024]
exports = [200, 190, 210, 220]

# Розрахунок середнього зростання на рік
annual_growth = []
for i in range(1, len(exports)):
    growth = (exports[i] - exports[i-1]) / exports[i-1] * 100
    annual_growth.append(growth)

# Середнє зростання
average_growth = sum(annual_growth) / len(annual_growth)

# Прогноз на 2025 на основі середнього зростання
forecast_2025 = exports[-1] * (1 + average_growth / 100)

print("Середнє зростання (%):", average_growth)
print("Базовий прогноз на 2025 (тон):", forecast_2025)`


Середнє зростання (%): 3.4294068504594812
Базовий прогноз на 2025 (тон): 227.54469507101084

Invoking: `Search` with `прогноз погоди для регіонів вирощування апельсинів у Бразилії на 2025 рік`


['The projected drop rate for the 2025–26 harvest is 20%, which is 2.2% higher than the previous harvest. This projection is related

{'input': '\n      Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.\n    ',
 'output': "### Прогноз експорту апельсинів з Бразилії на 2025 рік\n\n#### 1. **Аналіз історичних даних**\nІсторичні дані про експорт апельсинів з Бразилії за останні роки:\n- 2021: 200 тонн\n- 2022: 190 тонн\n- 2023: 210 тонн\n- 2024: 220 тонн\n\nНа основі цих даних було розраховано **середнє зростання експорту на рік**, яке склало **3.43%**. Виходячи з цього, базовий прогноз на 2025 рік становить **227.5 тонн**.\n\n---\n\n#### 2. **Зовнішні фактори**\n\n##### а) **Погодні умови в Бразилії**\nЗгідно з прогнозами, у 2025 році очікується **збільшення врожаю апельсинів на 19%** порівняно з попереднім роком. Це пов'язано зі сприятливими погодними умовами в ключових регіонах ви

Незрозумілі розрахунки. Агент знайшов багато даних, але не показав, як саме він дійшов до висновків +19% та +2%.

Він знайшов інформацію про зростання врожаю на 24% і одночасно про зменшення опадів на 15%. У фінальній відповіді він просто написав: "очікується збільшення врожаю апельсинів на 19%".

Також агент знайшов дані про зростання ВВП 3% та інфляцію 4.3%. Як з цього вийшло фінальне коригування +2%? Це також незрозуміло.

Тобто напевно потрібно оновити промпт, щоб агент показував всі свої розрахунки.

Можливо також попросити його зібрати історичні дані та зовнішньоекономічні фактори і використовувати методи ML для прогнозування експорту)
